In [1]:
import pandas as pd
import sys, os

sys.path.append(os.path.dirname(sys.path[0]))

In [2]:
from pathlib import Path


def _proj_root():
    """Localiza a raiz do projeto a partir do diretório atual (robusto a onde o kernel roda)."""
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "data" / "processed" / "neighbors.csv").exists():
            return c
    raise FileNotFoundError(
        "neighbors.csv não encontrado em data/processed/ a partir de %s" % Path.cwd()
    )


ROOT = _proj_root()
reader = pd.read_csv(ROOT / "data" / "processed" / "neighbors.csv")

In [3]:
reader

,query_uid,query_name,query_role,aspect,metric,rank,neighbor_uid,neighbor_name,neighbor_role,score,score_kind
0,2002078863,Scott King,AM,T,Euclidiana,1,19399448,Matheus Régis,AM,1.377396,distance
1,2002078863,Scott King,AM,T,Euclidiana,2,2002075081,Ricardo Cervantes,AM,1.378118,distance
2,2002078863,Scott King,AM,T,Euclidiana,3,31042062,Abraham Odoh,MID,1.539899,distance
3,2002078863,Scott King,AM,T,Euclidiana,4,2002078072,George Smith,AM,1.579829,distance
4,2002078863,Scott King,AM,T,Euclidiana,5,70109065,Fıratcan Üzüm,MID,1.649326,distance
...,...,...,...,...,...,...,...,...,...,...,...
9393955,37001809,Jordy Clasie,DEF,SP,Pearson,6,55063818,Pipas,MID,0.999914,similarity
9393956,37001809,Jordy Clasie,DEF,SP,Pearson,7,28127116,Tyler Onyango,DM,0.999866,similarity
9393957,37001809,Jordy Clasie,DEF,SP,Pearson,8,43190957,Antonio D'Alena,DM,0.999866,similarity
9393958,37001809,Jordy Clasie,DEF,SP,Pearson,9,2000062196,Tenner,DM,0.999866,similarity


In [4]:
neymar = reader.loc[reader['query_name']== 'Neymar']

In [5]:
neymar

,query_uid,query_name,query_role,aspect,metric,rank,neighbor_uid,neighbor_name,neighbor_role,score,score_kind
503570,19024412,Neymar,MID,T,Euclidiana,1,862068,Haris Medunjanin,DM,1.650114,distance
503571,19024412,Neymar,MID,T,Euclidiana,2,8430773,Jérémy Ménez,AM,1.806950,distance
503572,19024412,Neymar,MID,T,Euclidiana,3,14044150,Paulo Dybala,AM,1.836613,distance
503573,19024412,Neymar,MID,T,Euclidiana,4,8832853,Marcelo,DEF,1.943207,distance
503574,19024412,Neymar,MID,T,Euclidiana,5,85139014,Kylian Mbappé,AM,1.990651,distance
...,...,...,...,...,...,...,...,...,...,...,...
9114705,19024412,Neymar,MID,SP,Pearson,6,29023466,James Rowe,DM,0.999796,similarity
9114706,19024412,Neymar,MID,SP,Pearson,7,43039014,Felipe,DM,0.999796,similarity
9114707,19024412,Neymar,MID,SP,Pearson,8,25034173,Lukáš Masopust,DEF,0.999787,similarity
9114708,19024412,Neymar,MID,SP,Pearson,9,19028050,Thiago Carleto,DEF,0.999629,similarity


## Benchmark externo — distâncias a Messi vs. *Table 1* da referência

A **Table 1** da referência ranqueia 28 jogadores pela distância (Euclidiana) a **Lionel Messi**. Aqui comparamos esse ranking com o que o **SPSS** produz para Messi (`UID 7458500`, função `AM`), sobre os mesmos dados z-score **por posição** do experimento.

**Método.** O `neighbors.csv` guarda só o *top-10* por (aspecto × métrica) e não cobre os 28 nomeados; então calculamos a distância de Messi a **todos** os jogadores diretamente de `outfield_z.csv` — o mesmo que o `Retriever`/`distances.py` faz internamente (Euclidiana = L2 nas colunas z; verificado: o top-10 calculado aqui bate célula a célula com o `neighbors.csv`) — e localizamos cada jogador da Table 1 nesse ranking.

**Ressalvas.** (1) Similaridade não tem *ground truth*; a comparação é relativa — correlação de ranking, sobreposição e leitura qualitativa, não acerto absoluto. (2) Dois jogadores da Table 1 **não existem** neste FM2023 e entram como `AUSENTE`: *Agüero* (aposentado em dez/2021) e *C. Ronaldo* (fora desta base do Kaggle).

In [6]:
import json

import numpy as np
from scipy.stats import spearmanr, kendalltau

DATA = ROOT / "data" / "processed"

z = pd.read_csv(DATA / "outfield_z.csv")
raw = pd.read_csv(DATA / "outfield_raw.csv")
meta = (
    pd.read_csv(ROOT / "data" / "merged_players (1).csv", low_memory=False)
    .drop_duplicates("UID")
    .set_index("UID")[["Name", "Club", "Position", "Age"]]
)
feat = json.load(open(DATA / "feature_sets.json"))

MESSI = 7458500
Tcols = feat["technical"]
allcols = [c for c in z.columns if c not in ("UID", "primary_role")]
zU, rawU = z.set_index("UID"), raw.set_index("UID")
present = set(zU.index)
N = len(zU) - 1
# z-score GLOBAL (não por posição) — usado para isolar o efeito da normalização por função
gz = (rawU[allcols] - rawU[allcols].mean()) / rawU[allcols].std(ddof=0)


def messi_euclid(frame, cols):
    M = frame[cols].to_numpy(float)
    v = frame.loc[MESSI, cols].to_numpy(float)
    return pd.Series(np.sqrt(((M - v) ** 2).sum(1)), index=frame.index)


def messi_cosine(frame, cols, center):
    M = frame[cols].to_numpy(float)
    v = frame.loc[MESSI, cols].to_numpy(float)
    if center:  # Pearson = cosseno sobre vetores centrados
        M = M - M.mean(1, keepdims=True)
        v = v - v.mean()
    Mn = M / np.linalg.norm(M, axis=1, keepdims=True)
    return pd.Series(1 - Mn @ (v / np.linalg.norm(v)), index=frame.index)


DIST = {
    "perrole_full": messi_euclid(zU, allcols),   # primária: z por posição · 36 atributos
    "perrole_T":    messi_euclid(zU, Tcols),      # z por posição · Técnico (10)
    "global_full":  messi_euclid(gz, allcols),    # z GLOBAL · 36 atributos
    "cosseno_full": messi_cosine(zU, allcols, False),
    "pearson_full": messi_cosine(zU, allcols, True),
}
RANK = {k: v.drop(MESSI).rank(method="min") for k, v in DIST.items()}

# Table 1 da referência: (rank, jogador, distância, UID no nosso dataset | None se ausente)
TABLE1 = [
    (1, "Coutinho", 3.769, 19046041),   (2, "Hazard", 4.069, 18004418),     (3, "Thauvin", 4.140, 85046587),
    (4, "Dybala", 4.254, 14044150),     (5, "Aspas", 4.621, 67010455),      (6, "Aguero", 4.705, None),
    (7, "Salah", 4.849, 98028755),      (8, "Sterling", 4.858, 28054109),   (9, "Mbappe", 4.910, 85139014),
    (10, "Neymar", 4.945, 19024412),    (11, "A. Sanchez", 5.050, 75000261),(12, "L. Alberto", 5.126, 67131771),
    (13, "Immobile", 5.145, 43010773),  (14, "Mertens", 5.147, 8169425),    (15, "Di Maria", 5.247, 14000219),
    (16, "Kane", 5.363, 28049320),      (17, "Mahrez", 5.535, 85078058),    (18, "L. Suarez", 5.725, 78000335),
    (19, "Cavani", 5.744, 78005742),    (20, "Griezmann", 5.788, 67086656), (21, "C. Ronaldo", 6.000, None),
    (22, "N. Fekir", 6.067, 29114975),  (23, "De Bruyne", 6.414, 18004457), (24, "D. Silva", 6.526, 7458280),
    (25, "Aubameyang", 6.548, 43017197),(26, "Firmino", 6.826, 19068857),   (27, "Mariano", 7.246, 67157152),
    (28, "Lukaku", 7.489, 18007344),
]


def _row(rk, lab, dist, uid):
    if uid not in present:
        return [rk, lab, dist, "AUSENTE", np.nan, np.nan, np.nan, np.nan]
    return [
        rk, lab, dist, zU.loc[uid, "primary_role"],
        round(float(DIST["perrole_full"][uid]), 3), int(RANK["perrole_full"][uid]),
        round(float(DIST["perrole_T"][uid]), 3),     int(RANK["perrole_T"][uid]),
    ]


comp = pd.DataFrame(
    [_row(*t) for t in TABLE1],
    columns=["rank_T1", "jogador", "dist_T1", "role_nosso",
             "d_perrole_full", "rank_global_full", "d_T", "rank_global_T"],
)
print(f"Messi: UID={MESSI} · função={zU.loc[MESSI, 'primary_role']} · base = {N} jogadores de linha")
comp

Messi: UID=7458500 · função=AM · base = 78282 jogadores de linha


,rank_T1,jogador,dist_T1,role_nosso,d_perrole_full,rank_global_full,d_T,rank_global_T
0,1,Coutinho,3.769,MID,7.223,23.0,3.607,28.0
1,2,Hazard,4.069,AM,6.651,9.0,3.615,29.0
2,3,Thauvin,4.140,AM,8.166,95.0,4.708,207.0
3,4,Dybala,4.254,AM,5.916,2.0,2.488,3.0
4,5,Aspas,4.621,AM,8.375,127.0,5.236,520.0
5,6,Aguero,4.705,AUSENTE,NaN,NaN,NaN,NaN
6,7,Salah,4.849,AM,8.300,115.0,3.462,24.0
7,8,Sterling,4.858,MID,11.661,5980.0,6.585,4008.0
8,9,Mbappe,4.910,AM,7.389,28.0,3.375,21.0
9,10,Neymar,4.945,MID,5.138,1.0,2.382,2.0


In [7]:
# Correlação de ranking, convergência interna e sobreposição (somente nos jogadores presentes)
common = [(lab, dist, uid) for rk, lab, dist, uid in TABLE1 if uid in present]
t1_dist = np.array([d for _, d, _ in common])
uids = [u for _, _, u in common]
print(f"Jogadores da Table 1 presentes no FM2023: {len(common)}/28  (ausentes: Aguero, C. Ronaldo)\n")

print(f"Correlação de ranking — nossa distância a Messi  vs.  Table 1  (n={len(common)}):")
for k, nome in [
    ("perrole_full", "z por posição · full-36 (Euclid)  [primária]"),
    ("perrole_T",    "z por posição · Técnico-10 (Euclid)"),
    ("global_full",  "z GLOBAL · full-36 (Euclid)"),
    ("cosseno_full", "z por posição · full-36 (Cosseno)"),
    ("pearson_full", "z por posição · full-36 (Pearson)"),
]:
    o = np.array([DIST[k][u] for u in uids])
    sp, kt = spearmanr(t1_dist, o).statistic, kendalltau(t1_dist, o).statistic
    print(f"  {nome:44s} Spearman={sp:+.3f}  Kendall={kt:+.3f}")

print("\nConvergência interna das nossas 3 distâncias (nos 26 jogadores):")
for a, b in [("perrole_full", "perrole_T"), ("perrole_full", "cosseno_full"), ("perrole_full", "pearson_full")]:
    s = spearmanr([DIST[a][u] for u in uids], [DIST[b][u] for u in uids]).statistic
    print(f"  {a:13s} vs {b:13s} Spearman={s:+.3f}")

print(f"\nQuantos dos {len(uids)} jogadores da Table 1 caem perto de Messi no NOSSO ranking (de {N}):")
for k in ["perrole_full", "perrole_T"]:
    cs = "  ".join(f"top-{K}: {int(sum(RANK[k][u] <= K for u in uids))}" for K in (10, 50, 100, 500, 1000))
    print(f"  [{k:12s}] {cs}")

Jogadores da Table 1 presentes no FM2023: 26/28  (ausentes: Aguero, C. Ronaldo)

Correlação de ranking — nossa distância a Messi  vs.  Table 1  (n=26):
  z por posição · full-36 (Euclid)  [primária] Spearman=+0.449  Kendall=+0.317
  z por posição · Técnico-10 (Euclid)          Spearman=+0.432  Kendall=+0.280
  z GLOBAL · full-36 (Euclid)                  Spearman=+0.449  Kendall=+0.335
  z por posição · full-36 (Cosseno)            Spearman=+0.504  Kendall=+0.354
  z por posição · full-36 (Pearson)            Spearman=+0.507  Kendall=+0.360

Convergência interna das nossas 3 distâncias (nos 26 jogadores):
  perrole_full  vs perrole_T     Spearman=+0.919
  perrole_full  vs cosseno_full  Spearman=+0.990
  perrole_full  vs pearson_full  Spearman=+0.977

Quantos dos 26 jogadores da Table 1 caem perto de Messi no NOSSO ranking (de 78282):
  [perrole_full] top-10: 5  top-50: 11  top-100: 13  top-500: 18  top-1000: 18
  [perrole_T   ] top-10: 2  top-50: 12  top-100: 15  top-500: 17  top-1000:

In [8]:
# Quem o SPSS realmente devolve como mais parecido com Messi (entre 78.282 jogadores de linha)
def messi_top(series, n=15):
    t = series.drop(MESSI).sort_values().head(n)
    out = meta.loc[t.index, ["Name", "Club", "Position"]].copy()
    out.insert(0, "role", zU.loc[t.index, "primary_role"].to_numpy())
    out["dist"] = t.round(3).to_numpy()
    return out.reset_index(drop=True)


print("Top-15 vizinhos de Messi — z por posição · full-36 (Euclid):")
display(messi_top(DIST["perrole_full"]))

print("Top-15 vizinhos de Messi — z por posição · Técnico-10 (Euclid):")
display(messi_top(DIST["perrole_T"]))

# Conferência: o top-10 ARMAZENADO em neighbors.csv (aspecto T · Euclidiana) deve bater com o de cima
stored = (
    reader[(reader.query_uid == MESSI) & (reader.aspect == "T") & (reader.metric == "Euclidiana")]
    .sort_values("rank")[["rank", "neighbor_name", "neighbor_role", "score"]]
    .reset_index(drop=True)
)
print("Top-10 de Messi conforme ARMAZENADO em neighbors.csv (aspecto T · Euclidiana):")
display(stored)

Top-15 vizinhos de Messi — z por posição · full-36 (Euclid):


,role,Name,Club,Position,dist
0,MID,Neymar,PSG,"M (L), AM (LC), ST (C)",5.138
1,AM,Paulo Dybala,AS Roma,"AM (RC), ST (C)",5.916
2,FW,Wissam Ben Yedder,AS Monaco,ST (C),6.238
3,MID,Ángel Di María,Juventus,"M (RL), AM (RLC)",6.344
4,DEF,Álex Grimaldo,Benfica,D/WB/M (L),6.413
5,AM,Dimitri Payet,OM,AM (RLC),6.435
6,MID,Alejandro Gómez,Sevilla,"M (C), AM (LC), ST (C)",6.598
7,MID,Jonathan Viera,Las Palmas,M/AM (LC),6.649
8,AM,Eden Hazard,R. Madrid,AM (RLC),6.651
9,AM,Nabil Fekir,Real Hispalis,"AM (RLC), ST (C)",6.687


Top-15 vizinhos de Messi — z por posição · Técnico-10 (Euclid):


,role,Name,Club,Position,dist
0,DEF,Marcelo,Olympiacos,D/WB/M/AM (L),2.186
1,MID,Neymar,PSG,"M (L), AM (LC), ST (C)",2.382
2,AM,Paulo Dybala,AS Roma,"AM (RC), ST (C)",2.488
3,DEF,João Cancelo,Man City,"D/WB (RL), M (R)",2.603
4,DEF,Álex Grimaldo,Benfica,D/WB/M (L),2.646
5,DEF,Daniel Alves,Pumas,"D/WB (R), M (RC)",2.947
6,MID,Isco,Sevilla,M/AM (RLC),2.985
7,DEF,Raphaël Guerreiro,Borussia Dortmund,"D/WB (L), M (C)",3.004
8,AM,Marco Asensio,R. Madrid,AM (RLC),3.049
9,FW,Karim Benzema,R. Madrid,ST (C),3.058


Top-10 de Messi conforme ARMAZENADO em neighbors.csv (aspecto T · Euclidiana):


,rank,neighbor_name,neighbor_role,score
0,1,Marcelo,DEF,2.185605
1,2,Neymar,MID,2.381586
2,3,Paulo Dybala,AM,2.487919
3,4,João Cancelo,DEF,2.602899
4,5,Álex Grimaldo,DEF,2.646217
5,6,Daniel Alves,DEF,2.946646
6,7,Isco,MID,2.984660
7,8,Raphaël Guerreiro,DEF,3.004155
8,9,Marco Asensio,AM,3.048531
9,10,Karim Benzema,FW,3.057888


### Leitura dos resultados

**1. No topo, forte acordo.** O vizinho nº 1 de Messi no nosso sistema (z por posição, 36 atributos) é **Neymar**, seguido de **Dybala**, **Di María** e **Hazard** — todos no topo da Table 1 também. Os criadores ofensivos de elite são reconhecidos pelas duas metodologias.

**2. No ranking global, acordo apenas moderado** (Spearman ≈ **0,45**; Kendall ≈ **0,32**). As listas concordam no *tipo* de jogador, mas divergem bastante na ordem.

**3. O que causa a divergência: a normalização por posição.** A Table 1 mede proximidade *absoluta* de atributos; o SPSS mede estilo *dentro da função*. Por isso os **centroavantes puros** (`FW`), que a Table 1 mantém no meio da lista, são empurrados para **milhares de posições** no nosso ranking: Immobile (T1 #13 → nosso ~9277º), Cavani (#19 → ~6910º), Firmino (#26 → ~9730º), Mariano (#27 → ~8706º), Lukaku, Kane, Aubameyang. Messi é padronizado *entre os AMs*; um camisa 9 é padronizado *entre os 9s* — "finalizador de elite entre atacantes" ≠ "armador de elite entre meias-atacantes". Pelo mesmo motivo Coutinho, o nº 1 da Table 1, cai para ~23º (classificado `MID` aqui). Trocar a normalização por uma **global** (linha `global_full`) quase não muda a correlação (0,449) — confirmando que é o *z-score por posição*, e não um detalhe de implementação, que separa as duas visões.

**4. As nossas três distâncias convergem fortemente entre si** (Spearman 0,92–0,99), e Cosseno/Pearson ficam um pouco mais perto da Table 1 (≈0,50) que a Euclidiana (≈0,45) — responde à sub-pergunta 3: para esta consulta, escolher entre Euclidiana, Cosseno e Pearson quase não altera o resultado.

**5. Diferença de base.** Dois nomes da Table 1 não existem neste FM2023 — *Agüero* e *C. Ronaldo* —, indício de que a referência usa dados de uma temporada anterior. Comparar valores *absolutos* de distância entre os dois estudos não faz sentido (escalas e bases distintas); só a **estrutura de ranking** é comparável.

**Conclusão.** "Ser similar a Messi" no SPSS significa **ser tão atípico dentro da própria função quanto Messi é dentro da dele** — não ter atributos numericamente próximos dos dele. Isso explica tanto o acordo no topo (outros armadores excepcionais) quanto a divergência no corpo da lista (atacantes de área somem) em relação à Table 1.